# Manufacturing footprint inputs and itemised input costs

This notebook builds one Excel workbook with **two sheets**:

1. `manufacturing_inputs` — input-side manufacturing inventory from the SuperPro **SR** report.
2. `itemised_costs` — itemised input/cost rows from the SuperPro **ICR** report, covering materials, consumables/SUT, utilities and waste treatment lines where itemised by SuperPro.

The aim is to keep this separate from the output/waste-routing inventory. This notebook is for the **manufacturing footprint**: WFI and process chemicals, air, buffers, utilities such as electricity/steam/cooling water, and SUT/consumable costs.

Important accounting rule: the SR stream-detail table does not expose every input as an explicit `Source = INPUT` stream, especially cleaning/CIP-related additions. Therefore this notebook keeps explicit stream-detail input rows, then adds SR overall-component input reconciliation rows so the component-level input inventory closes back to SuperPro's overall input balance.

In [ ]:
from pathlib import Path
from datetime import date
import re
import math
import warnings

import pandas as pd

pd.set_option('display.max_columns', 160)
pd.set_option('display.width', 260)
pd.set_option('display.max_colwidth', 180)

RUN_DATE = date.today().strftime('%Y%m%d')

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name.lower() == 'notebooks' else Path.cwd()
RAW_DIR = PROJECT_ROOT / 'rawSP_reports'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
INTERIM_DIR = PROJECT_ROOT / 'data' / 'interim'

if not RAW_DIR.is_dir():
    raise FileNotFoundError(f'Supplied raw SuperPro reports directory is missing: {RAW_DIR}')
SEARCH_DIRS = [RAW_DIR]

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
INTERIM_DIR.mkdir(parents=True, exist_ok=True)

print('PROJECT_ROOT:', PROJECT_ROOT)
print('RAW_DIR:', RAW_DIR)
print('PROCESSED_DIR:', PROCESSED_DIR)
print('Search dirs:')
for p in SEARCH_DIRS:
    print(' ', p)


## 1. Locate and load the minimum SuperPro source files

This notebook only requires:

- **SR** / Materials & Streams Report: input stream detail and overall input component balance.
- **ICR** / Itemized Cost Report: itemised material, consumable/SUT, utility and waste-treatment costs.

No EIR, EMS, EQR or IDR file is used here.

In [ ]:
def find_report(label, patterns):
    matches = []
    # Prefer converted xlsx if present because it is easier to read without xlrd.
    ordered_patterns = []
    for p in patterns:
        if p.endswith('.xlsx'):
            ordered_patterns.append(p)
    for p in patterns:
        if p not in ordered_patterns:
            ordered_patterns.append(p)

    for directory in SEARCH_DIRS:
        for pattern in ordered_patterns:
            matches.extend(sorted(directory.glob(pattern)))
    matches = list(dict.fromkeys(matches))
    if len(matches) != 1:
        raise ValueError(f'Expected exactly one supplied {label} in {RAW_DIR}; found {matches}')
    print('Using:', matches[0])
    return matches[0]

SR_PATH = find_report('SR report', ['*SR*.xlsx', '*SR*.xls'])
ICR_PATH = find_report('ICR report', ['*ICR*.xlsx', '*ICR*.xls'])


In [ ]:
def cell_to_text(x):
    if x is None:
        return ''
    if isinstance(x, float) and math.isnan(x):
        return ''
    return str(x).strip()


def clean_text(x):
    s = cell_to_text(x).replace('\xa0', ' ').replace('\n', ' ')
    s = re.sub(r'\s+', ' ', s)
    return s.strip()


def to_number(x):
    s = clean_text(x)
    if s == '' or s.lower() in {'nan', 'none', '(none)', 'unknown'}:
        return None
    s = s.replace(',', '').replace('−', '-')
    try:
        return float(s)
    except ValueError:
        return None


def load_first_sheet(path):
    """Load SuperPro's single-sheet report export into a DataFrame of strings.

    Supports .xlsx through openpyxl and .xls through xlrd if xlrd is installed.
    If your local SuperPro reports are .xls and this fails, install xlrd in the venv:
        pip install xlrd
    """
    path = Path(path)
    if path.suffix.lower() == '.xlsx':
        from openpyxl import load_workbook
        wb = load_workbook(path, read_only=True, data_only=True)
        ws = wb[wb.sheetnames[0]]
        rows = []
        for row in ws.iter_rows(values_only=True):
            rows.append([cell_to_text(v) for v in row])
        return pd.DataFrame(rows), ws.title

    if path.suffix.lower() == '.xls':
        try:
            import xlrd
        except ImportError as exc:
            raise ImportError(
                'Reading .xls SuperPro reports requires xlrd. Either install xlrd or convert the SR/ICR reports to .xlsx.'
            ) from exc
        book = xlrd.open_workbook(str(path), formatting_info=False)
        sheet = book.sheet_by_index(0)
        rows = []
        for r in range(sheet.nrows):
            rows.append([cell_to_text(sheet.cell_value(r, c)) for c in range(sheet.ncols)])
        return pd.DataFrame(rows), sheet.name

    raise ValueError(f'Unsupported report file type: {path.suffix}')


def find_row_containing(df, text, start=0, end=None, col=None):
    if end is None:
        end = len(df)
    needle = text.lower()
    cols = [col] if col is not None else range(df.shape[1])
    for r in range(start, min(end, len(df))):
        for c in cols:
            if c < df.shape[1] and needle in clean_text(df.iloc[r, c]).lower():
                return r
    return None


def parse_unit_from_header(header_text):
    match = re.search(r'\(([^()]*)\)', clean_text(header_text))
    return match.group(1) if match else ''

sr_df, sr_sheet = load_first_sheet(SR_PATH)
icr_df, icr_sheet = load_first_sheet(ICR_PATH)

print('SR:', SR_PATH.name, sr_df.shape, 'sheet:', sr_sheet)
print('ICR:', ICR_PATH.name, icr_df.shape, 'sheet:', icr_sheet)


## 2. Process basis and procedure mapping

The annualisation uses the SR process basis. The procedure mapping is deliberately lightweight and only used for readable process-area labels. The quantities themselves come directly from SuperPro.

In [ ]:
def extract_process_basis_from_sr():
    start = find_row_containing(sr_df, '1. OVERALL PROCESS DATA')
    end = find_row_containing(sr_df, '2.1 STARTING MATERIAL REQUIREMENTS', start=start + 1)
    records = []
    for r in range(start + 1, end):
        label = clean_text(sr_df.iloc[r, 1]) if sr_df.shape[1] > 1 else ''
        if not label:
            continue
        value = None
        unit = ''
        for c in range(2, sr_df.shape[1]):
            parsed = to_number(sr_df.iloc[r, c])
            if parsed is not None:
                value = parsed
                for ucol in range(c + 1, sr_df.shape[1]):
                    maybe_unit = clean_text(sr_df.iloc[r, ucol])
                    if maybe_unit and to_number(maybe_unit) is None:
                        unit = maybe_unit
                        break
                break
        if value is not None:
            records.append({'parameter': label, 'value': value, 'unit': unit, 'source_row': r + 1})
    return pd.DataFrame(records)

basis_df = extract_process_basis_from_sr()
display(basis_df)

batches_per_year = float(basis_df.loc[basis_df['parameter'].str.contains('Number of Batches', case=False), 'value'].iloc[0])
batch_size_kg_mp = float(basis_df.loc[basis_df['parameter'].str.contains('Batch Size', case=False), 'value'].iloc[0])

print('batches_per_year:', batches_per_year)
print('batch_size_kg_mp:', batch_size_kg_mp)


def per_kg_mp(amount):
    if amount is None or pd.isna(amount) or batch_size_kg_mp == 0:
        return None
    return amount / batch_size_kg_mp


In [ ]:
SECTION_NAMES = {
    'Inoculum Prep', 'Cell Culture', 'Buffer Prep and Hold', 'Primary Recovery', 'Protein A',
    'Viral Inactivation', 'HIC Chrom', 'Membrane Adsorption', 'Filling'
}

PROCEDURE_SECTION_MAP = {
    # Upstream inoculum
    'P-01': ('upstream', 'Inoculum Prep', 'Ampule'),
    'P-02': ('upstream', 'Inoculum Prep', 'T-Flask'),
    'P-03': ('upstream', 'Inoculum Prep', 'Roller Bottle'),
    'P-04': ('upstream', 'Inoculum Prep', 'Disposable Bioreactor 100 L'),
    'P-05': ('upstream', 'Inoculum Prep', 'Disposable Bioreactor 600 L'),
    # Cell culture / media prep around production bioreactor
    'P-06': ('upstream', 'Cell Culture', 'Media preparation'),
    'P-07': ('upstream', 'Cell Culture', 'Media filtration'),
    'P-08': ('upstream', 'Cell Culture', 'Media hold'),
    'P-09': ('upstream', 'Cell Culture', 'Production bioreactor'),
    'P-10': ('upstream', 'Cell Culture', 'Media mixing'),
    # Primary recovery
    'P-11': ('downstream', 'Primary Recovery', 'Centrifugation / CIP'),
    'P-13': ('downstream', 'Primary Recovery', 'Depth filtration / preflush'),
    'P-14': ('downstream', 'Primary Recovery', 'Harvest hold / cooling'),
    'P-17': ('downstream', 'Protein A', 'Protein A feed hold'),
    # Chromatography / buffer prep procedures
    'P-18': ('downstream', 'Protein A', 'Protein A chromatography'),
    'P-19': ('downstream', 'Protein A', 'Protein A product hold'),
    'P-20': ('downstream', 'Protein A', 'Protein A column operation'),
    'P-21': ('downstream', 'Membrane Adsorption', 'Membrane adsorption'),
    'P-22': ('downstream', 'Membrane Adsorption', 'Membrane adsorption hold'),
    'P-23': ('downstream', 'Membrane Adsorption', 'Membrane adsorption operation'),
    'P-24': ('downstream', 'Viral Inactivation', 'Viral inactivation'),
    'P-12': ('downstream', 'HIC Chrom', 'HIC chromatography'),
    'P-15': ('downstream', 'Protein A', 'Column storage'),
    'P-27': ('formulation_fill', 'Filling', 'Filling'),
}

# Buffer-preparation sub-procedures are material-prep activities even if their buffer is later used in a column.
BUFFER_PREP_PREFIXES = (
    'P-18A', 'P-18B', 'P-18C', 'P-18D', 'P-18E', 'P-18F',
    'P-20a', 'P-21A', 'P-21B', 'P-23a', 'P-12A'
)


def base_procedure(proc):
    proc = clean_text(proc)
    m = re.match(r'(P-\d+)', proc)
    return m.group(1) if m else proc


def map_procedure(proc):
    proc = clean_text(proc)
    if any(proc.startswith(prefix) for prefix in BUFFER_PREP_PREFIXES):
        return {
            'broad_process_area': 'downstream',
            'superpro_section': 'Buffer Prep and Hold',
            'unit_operation': 'Buffer preparation / hold',
            'area_mapping_basis': 'procedure prefix mapped to buffer preparation',
        }
    base = base_procedure(proc)
    area, section, unit = PROCEDURE_SECTION_MAP.get(base, ('review_needed', 'review_needed', 'review_needed'))
    return {
        'broad_process_area': area,
        'superpro_section': section,
        'unit_operation': unit,
        'area_mapping_basis': 'procedure mapping table' if area != 'review_needed' else 'not_mapped',
    }


## 3. Extract manufacturing inputs from SR stream details

This extracts all explicit streams where `Source = INPUT`. Rows are kept on two accounting bases:

- `input_stream_component`: component-level rows, suitable for LCA when the component is explicitly present in stream details.
- `input_stream_total`: stream-total rows, useful for validating the stream quantities but **not** to sum with component rows.


In [ ]:
def classify_input_flow(flow_name, component=None):
    s = f'{flow_name} {component or ""}'.lower()
    if 'wfi' in s or 'water' in s:
        return 'water_or_wfi'
    if 'air' in s or component in {'Nitrogen', 'Oxygen'}:
        return 'air_or_gas'
    if 'buffer' in s or any(k in s for k in ['tris', 'hepes', 'sodium chloride', 'phosphat', 'pbs']):
        return 'buffer_or_salt'
    if any(k in s for k in ['naoh', 'hydroxid', 'h3po4', 'phosphoric', 'acetic']):
        return 'chemical_or_solvent'
    if 'media' in s or 'biomass' in s:
        return 'media_or_biological'
    return 'other_process_input'


def extract_sr_input_streams():
    block_starts = [r for r in range(len(sr_df)) if clean_text(sr_df.iloc[r, 1]) == 'Stream Name']
    component_balance_start = find_row_containing(sr_df, '4. OVERALL COMPONENT BALANCE', col=1)

    component_rows = []
    total_rows = []

    for block_index, start in enumerate(block_starts):
        block_end = block_starts[block_index + 1] if block_index + 1 < len(block_starts) else component_balance_start
        component_header = None
        for r in range(start, block_end):
            if clean_text(sr_df.iloc[r, 1]) == 'Component Flowrates (kg/batch)':
                component_header = r
                break
        if component_header is None:
            continue

        for c in range(2, sr_df.shape[1]):
            stream_name = clean_text(sr_df.iloc[start, c])
            if not stream_name:
                continue
            source = clean_text(sr_df.iloc[start + 1, c])
            destination = clean_text(sr_df.iloc[start + 2, c])
            if source.upper() != 'INPUT':
                continue

            proc_map = map_procedure(destination)
            stream_total_kg = None
            stream_total_l = None
            stream_components = []

            r = component_header + 1
            while r < block_end:
                component = clean_text(sr_df.iloc[r, 1])
                if component == '':
                    break
                amount = to_number(sr_df.iloc[r, c])
                if component == 'TOTAL (kg/batch)':
                    stream_total_kg = amount
                elif component == 'TOTAL (L/batch)':
                    stream_total_l = amount
                elif amount is not None and abs(amount) > 0:
                    stream_components.append({'component': component, 'amount_per_batch': amount, 'source_row': r + 1})
                r += 1

            component_sum = sum(x['amount_per_batch'] for x in stream_components)
            total_difference = None if stream_total_kg is None else stream_total_kg - component_sum

            for comp in stream_components:
                amount = comp['amount_per_batch']
                component_rows.append({
                    'source_report': 'SR',
                    'source_table': '3. STREAM DETAILS',
                    'record_type': 'input_stream_component',
                    'inventory_role': 'manufacturing_input_lca_component_basis',
                    'accounting_basis': 'stream_component_mass; safe LCA basis for explicit input streams',
                    **proc_map,
                    'source': source,
                    'destination_procedure': destination,
                    'flow_name_superpro': stream_name,
                    'input_category': classify_input_flow(stream_name, comp['component']),
                    'component': comp['component'],
                    'amount_per_batch': amount,
                    'unit_per_batch': 'kg/batch',
                    'amount_per_year': amount * batches_per_year,
                    'unit_per_year': 'kg/year',
                    'amount_per_kg_mp': per_kg_mp(amount),
                    'unit_per_kg_mp': 'kg/kg MP',
                    'stream_total_kg_per_batch': stream_total_kg,
                    'stream_volume_L_per_batch': stream_total_l,
                    'stream_component_sum_kg_per_batch': component_sum,
                    'stream_total_minus_component_sum_kg_per_batch': total_difference,
                    'include_in_manufacturing_lca': 'yes_explicit_stream_component',
                    'mapping_status': 'explicit Source=INPUT stream component from SR stream details',
                    'notes': 'Component in an explicit SR stream where Source = INPUT. Use these rows for detailed input-stream LCA, but also include residual rows below to close to SR overall input balance.',
                    'source_row': comp['source_row'],
                    'source_column': c + 1,
                })

            total_rows.append({
                'source_report': 'SR',
                'source_table': '3. STREAM DETAILS',
                'record_type': 'input_stream_total',
                'inventory_role': 'input_stream_total_validation_only',
                'accounting_basis': 'stream_total; do not add to component rows',
                **proc_map,
                'source': source,
                'destination_procedure': destination,
                'flow_name_superpro': stream_name,
                'input_category': classify_input_flow(stream_name),
                'component': 'TOTAL',
                'amount_per_batch': stream_total_kg,
                'unit_per_batch': 'kg/batch',
                'amount_per_year': stream_total_kg * batches_per_year if stream_total_kg is not None else None,
                'unit_per_year': 'kg/year',
                'amount_per_kg_mp': per_kg_mp(stream_total_kg),
                'unit_per_kg_mp': 'kg/kg MP',
                'stream_total_kg_per_batch': stream_total_kg,
                'stream_volume_L_per_batch': stream_total_l,
                'stream_component_sum_kg_per_batch': component_sum,
                'stream_total_minus_component_sum_kg_per_batch': total_difference,
                'include_in_manufacturing_lca': 'no_validation_only',
                'mapping_status': 'explicit Source=INPUT stream total from SR stream details',
                'notes': 'Input stream total from SR. Included to validate component sums, not to add to component rows.',
                'source_row': start + 1,
                'source_column': c + 1,
            })

    return pd.DataFrame(component_rows), pd.DataFrame(total_rows)

sr_input_components_df, sr_input_totals_df = extract_sr_input_streams()
print('Explicit input stream component rows:', len(sr_input_components_df))
print('Explicit input stream total rows:', len(sr_input_totals_df))
print('Explicit Source=INPUT stream total kg/batch:', sr_input_totals_df['amount_per_batch'].sum())
print('Explicit Source=INPUT component sum kg/batch:', sr_input_components_df['amount_per_batch'].sum())
display(sr_input_totals_df[['destination_procedure','flow_name_superpro','input_category','amount_per_batch','stream_component_sum_kg_per_batch','stream_total_minus_component_sum_kg_per_batch']])


## 4. Extract SR overall component input balance and reconcile missing input components

This is the check that prevents the WFI mistake. Explicit `Source = INPUT` streams are not necessarily the whole manufacturing input burden. SuperPro's **overall component balance** is retained as the closure target.

Rows with `record_type = overall_minus_stream_input_residual` are not physical stream-detail rows, but they represent input component mass present in the SR overall input balance and not in explicit input streams. These are kept so manufacturing LCA can close to the full SuperPro input balance.

In [ ]:
def extract_sr_overall_component_inputs(include_zero=False):
    start = find_row_containing(sr_df, '4. OVERALL COMPONENT BALANCE')
    end = find_row_containing(sr_df, '5. EQUIPMENT CONTENTS', start=start + 1)
    
    header = None
    for rr in range(start, end):
        if clean_text(sr_df.iloc[rr, 1]).upper() == 'COMPONENT':
            header = rr
            break
    if header is None:
        raise ValueError('Could not locate COMPONENT header row in SR overall component balance.')

    header_labels = {clean_text(sr_df.iloc[header, c]).upper(): c for c in range(sr_df.shape[1]) if clean_text(sr_df.iloc[header, c])}
    input_col = header_labels.get('INPUT')
    if input_col is None:
        raise ValueError('Could not locate INPUT column in SR overall component balance header.')

    records = []
    for r in range(header + 1, end):
        component = clean_text(sr_df.iloc[r, 1])
        if not component or component.upper() == 'TOTAL' or component.startswith('NOTE:'):
            continue
        amount = to_number(sr_df.iloc[r, input_col])
        if amount is None:
            continue
        if not include_zero and abs(amount) <= 0:
            continue
        records.append({
            'source_report': 'SR',
            'source_table': '4. OVERALL COMPONENT BALANCE (kg/batch)',
            'record_type': 'overall_component_input',
            'inventory_role': 'overall_input_balance_validation_basis',
            'accounting_basis': 'overall_component_input_balance; do not add to stream rows',
            'broad_process_area': 'overall_process',
            'superpro_section': 'Entire Process',
            'unit_operation': 'Entire Process',
            'area_mapping_basis': 'SR overall component balance',
            'source': 'OVERALL_INPUT_BALANCE',
            'destination_procedure': '',
            'flow_name_superpro': component,
            'input_category': classify_input_flow(component, component),
            'component': component,
            'amount_per_batch': amount,
            'unit_per_batch': 'kg/batch',
            'amount_per_year': amount * batches_per_year,
            'unit_per_year': 'kg/year',
            'amount_per_kg_mp': per_kg_mp(amount),
            'unit_per_kg_mp': 'kg/kg MP',
            'stream_total_kg_per_batch': None,
            'stream_volume_L_per_batch': None,
            'stream_component_sum_kg_per_batch': None,
            'stream_total_minus_component_sum_kg_per_batch': None,
            'include_in_manufacturing_lca': 'no_validation_only',
            'mapping_status': 'overall component input balance from SR',
            'notes': 'Overall process input component balance. Used as closure target; do not sum with explicit stream rows.',
            'source_row': r + 1,
            'source_column': input_col + 1,
        })
    return pd.DataFrame(records)

sr_overall_input_df = extract_sr_overall_component_inputs(include_zero=False)

stream_by_component = sr_input_components_df.groupby('component', dropna=False)['amount_per_batch'].sum().rename('explicit_stream_input_kg_per_batch')
overall_by_component = sr_overall_input_df.groupby('component', dropna=False)['amount_per_batch'].sum().rename('overall_input_kg_per_batch')
input_reconciliation_df = pd.concat([overall_by_component, stream_by_component], axis=1).fillna(0).reset_index()
input_reconciliation_df['overall_minus_explicit_stream_kg_per_batch'] = input_reconciliation_df['overall_input_kg_per_batch'] - input_reconciliation_df['explicit_stream_input_kg_per_batch']
input_reconciliation_df['overall_minus_explicit_stream_kg_per_year'] = input_reconciliation_df['overall_minus_explicit_stream_kg_per_batch'] * batches_per_year
input_reconciliation_df = input_reconciliation_df.sort_values('overall_minus_explicit_stream_kg_per_batch', key=lambda s: s.abs(), ascending=False)

display(input_reconciliation_df)

residual_rows = []
for _, row in input_reconciliation_df.iterrows():
    diff = row['overall_minus_explicit_stream_kg_per_batch']
    if abs(diff) <= 0.005:
        continue
    residual_rows.append({
        'source_report': 'SR',
        'source_table': '4. OVERALL COMPONENT BALANCE minus 3. STREAM DETAILS',
        'record_type': 'overall_minus_stream_input_residual',
        'inventory_role': 'manufacturing_input_lca_residual_component_basis',
        'accounting_basis': 'overall_input_minus_explicit_stream_components; add to explicit components for full component input footprint',
        'broad_process_area': 'overall_process',
        'superpro_section': 'Overall input reconciliation',
        'unit_operation': 'Non-stream-detail input / cleaning / rounding reconciliation',
        'area_mapping_basis': 'calculated from SR overall component input balance and explicit Source=INPUT stream components',
        'source': 'INPUT_RECONCILIATION',
        'destination_procedure': '',
        'flow_name_superpro': row['component'],
        'input_category': classify_input_flow(row['component'], row['component']),
        'component': row['component'],
        'amount_per_batch': diff,
        'unit_per_batch': 'kg/batch',
        'amount_per_year': diff * batches_per_year,
        'unit_per_year': 'kg/year',
        'amount_per_kg_mp': per_kg_mp(diff),
        'unit_per_kg_mp': 'kg/kg MP',
        'stream_total_kg_per_batch': None,
        'stream_volume_L_per_batch': None,
        'stream_component_sum_kg_per_batch': row['explicit_stream_input_kg_per_batch'],
        'stream_total_minus_component_sum_kg_per_batch': None,
        'include_in_manufacturing_lca': 'yes_residual_to_close_overall_input_balance',
        'mapping_status': 'input component present in overall balance but not fully present in explicit Source=INPUT stream details',
        'notes': 'Include with explicit input stream components when the goal is the complete manufacturing input footprint. Likely includes cleaning/CIP additions or other SuperPro input accounting not exposed as normal Source=INPUT streams.',
        'source_row': '',
        'source_column': '',
    })

sr_input_residual_df = pd.DataFrame(residual_rows)
print('Residual component rows needed to close input balance:', len(sr_input_residual_df))
display(sr_input_residual_df[['component','amount_per_batch','amount_per_year','include_in_manufacturing_lca','notes']])


## 5. Build manufacturing input inventory sheet

In [ ]:
manufacturing_columns = [
    'inventory_id', 'source_report', 'source_table', 'record_type', 'inventory_role', 'accounting_basis',
    'broad_process_area', 'superpro_section', 'unit_operation', 'area_mapping_basis',
    'source', 'destination_procedure', 'flow_name_superpro', 'input_category', 'component',
    'amount_per_batch', 'unit_per_batch', 'amount_per_year', 'unit_per_year',
    'amount_per_kg_mp', 'unit_per_kg_mp', 'stream_total_kg_per_batch', 'stream_volume_L_per_batch',
    'stream_component_sum_kg_per_batch', 'stream_total_minus_component_sum_kg_per_batch',
    'include_in_manufacturing_lca', 'mapping_status', 'notes', 'source_row', 'source_column'
]

frames = [sr_input_components_df, sr_input_totals_df, sr_overall_input_df, sr_input_residual_df]
for frame in frames:
    for col in manufacturing_columns:
        if col not in frame.columns and col != 'inventory_id':
            frame[col] = None

manufacturing_inputs_df = pd.concat(frames, ignore_index=True, sort=False)
record_sort = {
    'input_stream_component': 1,
    'overall_minus_stream_input_residual': 2,
    'input_stream_total': 3,
    'overall_component_input': 4,
}
manufacturing_inputs_df['_sort'] = manufacturing_inputs_df['record_type'].map(record_sort).fillna(99)
manufacturing_inputs_df = manufacturing_inputs_df.sort_values(
    ['_sort', 'broad_process_area', 'superpro_section', 'destination_procedure', 'flow_name_superpro', 'component'],
    na_position='last'
).drop(columns=['_sort']).reset_index(drop=True)
manufacturing_inputs_df.insert(0, 'inventory_id', [f'MFG{n:04d}' for n in range(1, len(manufacturing_inputs_df) + 1)])
manufacturing_inputs_df = manufacturing_inputs_df[manufacturing_columns]

print('Manufacturing input inventory rows:', len(manufacturing_inputs_df))
display(manufacturing_inputs_df.head(80))


## 6. Extract itemised costs from the ICR

This uses the detailed ICR sections where SuperPro provides item-level quantities:

- `5.1.4 MATERIAL COST - DETAILED BREAKDOWN BY MATERIAL TYPE`
- `7.4 CONSUMABLES COST - DETAILED BREAKDOWN BY TYPE`
- `8.4 WASTE COST - DETAILED BREAKDOWN BY WASTE TYPE`
- `9.4 UTILITIES COST - DETAILED BREAKDOWN BY UTILITY TYPE`

Waste is included in this cost sheet for completeness because it is a real ICR operating-cost line, although it is not an input material for the manufacturing footprint.

In [ ]:
def find_header_columns(df, header_row):
    cols = {}
    for c in range(df.shape[1]):
        text = clean_text(df.iloc[header_row, c])
        upper = text.upper()
        if 'PROCEDURE' in upper or 'EQUIPMENT' in upper:
            cols['entity_col'] = c
        if 'UNIT COST' in upper:
            cols['unit_cost_col'] = c
            cols['unit_cost_unit'] = parse_unit_from_header(text)
        if 'AMOUNT' in upper:
            cols['amount_col'] = c
            cols['amount_unit'] = parse_unit_from_header(text)
        if upper.startswith('COST') or 'COST ($/YEAR)' in upper:
            cols['cost_col'] = c
        if upper == '%':
            cols['percent_col'] = c
    return cols


def extract_detailed_cost_section(cost_group, start_marker, end_marker=None):
    start = find_row_containing(icr_df, start_marker)
    end = find_row_containing(icr_df, end_marker, start=start + 1) if end_marker else len(icr_df)
    if start is None:
        raise ValueError(f'Could not find ICR section: {start_marker}')
    if end is None:
        end = len(icr_df)

    records = []
    current_item = None
    current_section = None
    in_table = False
    header_cols = {}

    for r in range(start + 1, end):
        label = clean_text(icr_df.iloc[r, 1]) if icr_df.shape[1] > 1 else ''
        if not label:
            continue

        if label.upper() == 'TOTAL':
            in_table = False
            continue

        is_header = any(k in label.upper() for k in ['PROCEDURE NAME', 'EQUIPMENT NAME', 'PROCEDURE / STREAM'])
        if is_header:
            header_cols = find_header_columns(icr_df, r)
            in_table = True
            continue

        if in_table:
            entity_col = header_cols.get('entity_col', 1)
            unit_cost_col = header_cols.get('unit_cost_col')
            amount_col = header_cols.get('amount_col')
            cost_col = header_cols.get('cost_col')
            percent_col = header_cols.get('percent_col')

            entity = clean_text(icr_df.iloc[r, entity_col])
            if not entity or entity.upper() == 'TOTAL':
                in_table = False
                continue

            unit_cost = to_number(icr_df.iloc[r, unit_cost_col]) if unit_cost_col is not None else None
            amount_per_year = to_number(icr_df.iloc[r, amount_col]) if amount_col is not None else None
            cost_per_year = to_number(icr_df.iloc[r, cost_col]) if cost_col is not None else None
            percent = to_number(icr_df.iloc[r, percent_col]) if percent_col is not None else None

            amount_unit = header_cols.get('amount_unit', '')
            if amount_unit in {'/year', ''} and amount_col is not None:
                # Some ICR tables put the unit in a separate column between Amount and Cost.
                for uc in range(amount_col + 1, cost_col or amount_col + 1):
                    candidate = clean_text(icr_df.iloc[r, uc])
                    if candidate and to_number(candidate) is None:
                        amount_unit = candidate.replace(' /year', '/year').strip()
                        if not amount_unit.endswith('/year'):
                            amount_unit = f'{amount_unit}/year'
                        break
            amount_unit = amount_unit.replace(' ', '')

            amount_per_batch = amount_per_year / batches_per_year if amount_per_year is not None else None
            cost_per_batch = cost_per_year / batches_per_year if cost_per_year is not None else None

            proc_map = map_procedure(entity.split('/')[0].strip()) if entity.startswith('P-') else {
                'broad_process_area': 'not_procedure_specific',
                'superpro_section': current_section or '',
                'unit_operation': entity,
                'area_mapping_basis': 'ICR section label',
            }

            records.append({
                'source_report': 'ICR',
                'source_table': start_marker,
                'cost_group': cost_group,
                'process_section': current_section,
                'item_name': current_item,
                'procedure_or_equipment': entity,
                'broad_process_area': proc_map['broad_process_area'],
                'superpro_section': proc_map['superpro_section'] if proc_map['superpro_section'] != 'review_needed' else current_section,
                'unit_operation': proc_map['unit_operation'],
                'area_mapping_basis': proc_map['area_mapping_basis'],
                'unit_cost': unit_cost,
                'unit_cost_unit': header_cols.get('unit_cost_unit', ''),
                'amount_per_year': amount_per_year,
                'amount_unit_per_year': amount_unit,
                'amount_per_batch': amount_per_batch,
                'amount_unit_per_batch': amount_unit.replace('/year', '/batch') if amount_unit else '',
                'cost_per_year_usd': cost_per_year,
                'cost_per_batch_usd': cost_per_batch,
                'percent_within_item_or_section': percent,
                'include_in_manufacturing_input_cost': 'yes' if cost_group in {'material', 'consumable', 'utility'} else 'no_output_or_waste_cost',
                'notes': 'Detailed itemised ICR row. Waste rows are included for cost completeness but are not manufacturing input materials.',
                'source_row': r + 1,
            })
            continue

        # Not inside a table: label is either an item/type name or a process section name.
        if label in SECTION_NAMES:
            current_section = label
        else:
            # Ignore section title echoes that accidentally appear in the scan.
            if not re.match(r'^\d+(\.\d+)*\s', label):
                current_item = label
                current_section = None

    return pd.DataFrame(records)

material_cost_df = extract_detailed_cost_section('material', '5.1.4 MATERIAL COST - DETAILED BREAKDOWN BY MATERIAL TYPE', '6.1 LABOR COST')
consumable_cost_df = extract_detailed_cost_section('consumable', '7.4 CONSUMABLES COST - DETAILED BREAKDOWN BY TYPE', '8.1 WASTE COST')
waste_cost_df = extract_detailed_cost_section('waste', '8.4 WASTE COST - DETAILED BREAKDOWN BY WASTE TYPE', '9.1 UTILITIES COST')
utility_cost_df = extract_detailed_cost_section('utility', '9.4 UTILITIES COST - DETAILED BREAKDOWN BY UTILITY TYPE', None)

itemised_costs_df = pd.concat([material_cost_df, consumable_cost_df, utility_cost_df, waste_cost_df], ignore_index=True, sort=False)
itemised_costs_df.insert(0, 'cost_id', [f'COST{n:04d}' for n in range(1, len(itemised_costs_df) + 1)])

print('Itemised cost rows:', len(itemised_costs_df))
display(itemised_costs_df.groupby('cost_group', dropna=False).agg(rows=('cost_id','count'), cost_per_year_usd=('cost_per_year_usd','sum'), amount_rows=('amount_per_year','count')).reset_index())
display(itemised_costs_df.head(100))


## 7. Validation checks

These checks deliberately separate accounting bases, so totals are only compared where the bases match.

In [ ]:
# 1. Explicit input stream totals should match explicit input stream component sums.
stream_closure = sr_input_totals_df.copy()
stream_closure['abs_difference'] = stream_closure['stream_total_minus_component_sum_kg_per_batch'].abs()
stream_closure_failures = stream_closure[stream_closure['abs_difference'] > 0.02]
print('Explicit input stream closure failures > 0.02 kg/batch:', len(stream_closure_failures))
if len(stream_closure_failures):
    display(stream_closure_failures[['flow_name_superpro','destination_procedure','amount_per_batch','stream_component_sum_kg_per_batch','stream_total_minus_component_sum_kg_per_batch']])

# 2. Component-level complete manufacturing input basis should close exactly to SR overall input balance.
lca_input_basis = manufacturing_inputs_df[manufacturing_inputs_df['include_in_manufacturing_lca'].isin([
    'yes_explicit_stream_component', 'yes_residual_to_close_overall_input_balance'
])]
complete_by_component = lca_input_basis.groupby('component', dropna=False)['amount_per_batch'].sum().rename('complete_input_basis_kg_per_batch')
overall_by_component = sr_overall_input_df.groupby('component', dropna=False)['amount_per_batch'].sum().rename('overall_input_balance_kg_per_batch')
complete_validation = pd.concat([overall_by_component, complete_by_component], axis=1).fillna(0).reset_index()
complete_validation['difference_kg_per_batch'] = complete_validation['overall_input_balance_kg_per_batch'] - complete_validation['complete_input_basis_kg_per_batch']
complete_validation = complete_validation.sort_values('difference_kg_per_batch', key=lambda s: s.abs(), ascending=False)
print('Max component closure difference after residual rows:', complete_validation['difference_kg_per_batch'].abs().max())
display(complete_validation)

# 3. WFI visibility check.
wfi_check = complete_validation[complete_validation['component'].str.contains('WFI|water', case=False, na=False)]
print('WFI / water rows in complete input validation:')
display(wfi_check)

# 4. Cost checks by detailed ICR group.
cost_group_validation = itemised_costs_df.groupby('cost_group', dropna=False).agg(
    rows=('cost_id','count'),
    amount_rows=('amount_per_year','count'),
    cost_per_year_usd=('cost_per_year_usd','sum'),
    cost_per_batch_usd=('cost_per_batch_usd','sum'),
).reset_index()
display(cost_group_validation)

# 5. Internal total from sheet should equal cost_per_year / batches_per_year.
itemised_costs_df['cost_batch_validation_difference'] = itemised_costs_df['cost_per_batch_usd'] - (itemised_costs_df['cost_per_year_usd'] / batches_per_year)
max_cost_batch_diff = itemised_costs_df['cost_batch_validation_difference'].abs().max()
print('Max cost_per_batch validation difference:', max_cost_batch_diff)

assert len(stream_closure_failures) == 0, 'Some explicit Source=INPUT stream totals do not match their component sums.'
assert complete_validation['difference_kg_per_batch'].abs().max() <= 1e-9, 'Complete input basis does not close to SR overall component input balance.'
assert max_cost_batch_diff <= 1e-9 or pd.isna(max_cost_batch_diff), 'Cost per batch validation failed.'

print('Validation passed.')


## 8. Export the two-sheet workbook

Only the two requested sheets are exported. Validation tables stay in the notebook output so the workbook remains a clean input file for LCA/optimisation.

In [ ]:
output_xlsx = PROCESSED_DIR / f'manufacturing_footprint_inputs_and_costs_{RUN_DATE}.xlsx'
output_csv_inputs = PROCESSED_DIR / f'manufacturing_inputs_{RUN_DATE}.csv'
output_csv_costs = PROCESSED_DIR / f'itemised_costs_{RUN_DATE}.csv'

manufacturing_inputs_df.to_csv(output_csv_inputs, index=False)
itemised_costs_df.to_csv(output_csv_costs, index=False)

with pd.ExcelWriter(output_xlsx, engine='openpyxl') as writer:
    manufacturing_inputs_df.to_excel(writer, sheet_name='manufacturing_inputs', index=False)
    itemised_costs_df.to_excel(writer, sheet_name='itemised_costs', index=False)

    from openpyxl.styles import Font, PatternFill, Alignment, Border, Side
    from openpyxl.utils import get_column_letter

    header_fill = PatternFill('solid', fgColor='1F4E78')
    header_font = Font(color='FFFFFF', bold=True)
    thin_blue = Side(style='thin', color='D9E2F3')

    width_overrides = {
        'manufacturing_inputs': {
            'A': 12, 'B': 12, 'C': 44, 'D': 34, 'E': 44, 'F': 58, 'G': 20, 'H': 28,
            'I': 34, 'J': 38, 'K': 14, 'L': 20, 'M': 32, 'N': 24, 'O': 24, 'P': 16,
            'Q': 16, 'R': 16, 'S': 16, 'T': 18, 'U': 16, 'V': 18, 'W': 18, 'X': 24,
            'Y': 28, 'Z': 30, 'AA': 50, 'AB': 70, 'AC': 14, 'AD': 14,
        },
        'itemised_costs': {
            'A': 12, 'B': 12, 'C': 44, 'D': 16, 'E': 28, 'F': 36, 'G': 36, 'H': 22,
            'I': 28, 'J': 34, 'K': 38, 'L': 14, 'M': 16, 'N': 16, 'O': 16, 'P': 16,
            'Q': 16, 'R': 16, 'S': 18, 'T': 20, 'U': 24, 'V': 60, 'W': 14, 'X': 18,
        },
    }

    for sheet_name in ['manufacturing_inputs', 'itemised_costs']:
        ws = writer.book[sheet_name]
        ws.freeze_panes = 'A2'
        ws.auto_filter.ref = ws.dimensions
        for cell in ws[1]:
            cell.fill = header_fill
            cell.font = header_font
            cell.alignment = Alignment(horizontal='center', vertical='center', wrap_text=True)
            cell.border = Border(bottom=thin_blue)
        for row in ws.iter_rows(min_row=2):
            for cell in row:
                cell.alignment = Alignment(vertical='top', wrap_text=True)
        for col_letter, width in width_overrides[sheet_name].items():
            ws.column_dimensions[col_letter].width = width
        for col in ws.iter_cols(min_row=2, max_row=ws.max_row):
            header = ws.cell(row=1, column=col[0].column).value or ''
            if any(k in str(header) for k in ['amount', 'cost', 'difference', 'total', 'volume', 'unit_cost', 'percent']):
                for cell in col:
                    cell.number_format = '0.0000'

print('Wrote Excel:', output_xlsx)
print('Wrote inputs CSV:', output_csv_inputs)
print('Wrote costs CSV:', output_csv_costs)
